## CFAR: 恒虚警检测(Constant False-Alarm Rate)

<p align="center">
  <img src="images/2025-03-16-17-07-52.png" width="60%">
</p>

### 参考资料

- [雷达信号处理中的恒虚警检测(CFAR)技术概述](https://blog.csdn.net/xhblair/article/details/138784323)

- [雷达信号处理之恒虚警（CFAR）检测基础知识总结](https://zhuanlan.zhihu.com/p/652220176)

### 目标SNR、虚警率、检出率之间的关系

In [2]:
! pip install marcumq 

In [1]:
import numpy as np  
import matplotlib.pyplot as plt  
import marcumq  

# -----------------------  
# 参数设置  
# -----------------------  
P_fa_values = [1e-2, 1e-4, 1e-6, 1e-8]  # 虚警概率  
SNR_dB = np.linspace(0, 18, 200)         # 信噪比(dB)  
SNR = 10 ** (SNR_dB / 10)                # 线性信噪比  

# -----------------------  
# 检测概率计算  
# -----------------------  
Pd_curves = []  
for P_fa in P_fa_values:  
    threshold = np.sqrt(-2 * np.log(P_fa))  # 根据 P_fa 计算门限  
    Pd = marcumq.marcumq(1, np.sqrt(2 * SNR), threshold)  # 检测概率公式  
    Pd_curves.append(Pd)  

# -----------------------  
# 标注 SNR = 10dB 点  
# -----------------------  
SNR_dB_target = 10  
SNR_target_idx = np.argmin(np.abs(SNR_dB - SNR_dB_target))  # 找到最接近10dB的索引  
annotations = []  # 存储每条曲线的标注信息  
for i, P_fa in enumerate(P_fa_values):  
    Pd_value = Pd_curves[i][SNR_target_idx]  # 对应虚警概率的检测概率  
    annotations.append((SNR_dB_target, Pd_value, f'$P_{{fa}} = {P_fa:.0e}, P_d = {Pd_value:.4f}$'))  


# -----------------------  
# 绘制 Pd-SNR 曲线  
# -----------------------  
plt.figure(figsize=(8, 6))  
colors = ['b', 'orange', 'g', 'purple']  

for i, P_fa in enumerate(P_fa_values):  
    plt.plot(SNR_dB, Pd_curves[i], label=f'$P_{{fa}} = {P_fa}$', color=colors[i])  
    # 标注 SNR = 10dB 的点  
    SNR_dB_target, Pd_value, text = annotations[i]  
    plt.scatter(SNR_dB_target, Pd_value, color=colors[i], label=None)  
    plt.text(SNR_dB_target + 0.5, Pd_value, f'{Pd_value:.4f}', color=colors[i], fontsize=10)  


plt.grid(True, linestyle='--', alpha=0.7)  
plt.xlabel('SNR (dB)', fontsize=12)  
plt.ylabel('Pd ($P_d$)', fontsize=12)  
plt.title('Pd and SNR', fontsize=14)  
plt.legend(fontsize=10)  
plt.xlim([0, 18])  
plt.ylim([0, 1])  
plt.show()  

ModuleNotFoundError: No module named 'marcumq'

从上面的仿真结果中可以得到的一些结论：

1、虚警率不变时，**检出率会随着目标SNR的增加而增加的，这呼应了我们之前对检出率的定义**，虚警率不变说明阈值不变，SNR增加表示曲线整体右移，于是检出率(阴影部分面积)自然增加。

2、如果我们使目标SNR不变，去看检出率与虚警率的变化关系(对应图中固定横坐标)：**检出率会随着虚警率的增加而增加：虚警率变大说明阈值变小了，阈值变小了自然检出率会增加，所以我们要想提高检出率，要么想办法提高目标的SNR，而如果在目标SNR没法提高时我们只能牺牲虚警率**。

3、接上面第2点，这条曲线只是告诉我们特定Pfa下目标SNR与检出率的关系，我们可以通过设计不同的Pfa来使得这条曲线上下移动，也即：由Pfa的变化导致的阈值的变化只是会影响同一个SNR下的检测概率发生变化，不是说设定某个阈值后阈值以下的SNR的目标就检测不出来了！

---

## 均值类CFAR门限因子

### CA-CFAR

对于 **CA-CFAR（Cell-Averaging CFAR）**，门限因子 $ T $的确定通常基于设定的虚警概率 $ P_{fa} $ 和参考单元数目 $ R $。其计算公式为：

$$
T = \left( P_{fa} \right)^{-1/R} - 1
$$

### 公式含义
- $ P_{fa} $：虚警概率（False Alarm Probability），表示没有目标的情况下信号超过门限的概率。
- $ R $：参考单元的数量（通常为窗口数据的个数）。
- $ T $：门限因子，用于调整检测门限的大小。

### 具体计算步骤
1. **给定参数：**
   - 确定虚警概率 $ P_{fa} $，例如 $ P_{fa} = 10^{-6} $
   - 确定参考单元数 $ R $（例如，若滑窗左右各有 4 个参考单元，则 $R = 8 $ )
   
2. **带入公式计算：**
   $$ 
   T = \left( P_{fa} \right)^{-1/R} - 1
   $$

   例如：
   - $ P_{fa} = 10^{-6} $
   - $  R = 8 $ 
   
   则：
   $$
   T = \left( 10^{-6} \right)^{-1/8} - 1 = 4.6234
   $$

### 实现代码示例（Python）
以下是用 Python 计算 CA-CFAR 中门限因子的简单代码实现：


In [3]:
import numpy as np

# 参数设置
P_fa = 1e-6  # 虚警概率
R = 8       # 参考单元数目

# 计算门限因子 T
T = (P_fa ** (-1 / R)) - 1
print(f"CA-CFAR 的门限因子 T = {T:.4f}")

CA-CFAR 的门限因子 T = 4.6234



### 总结
门限因子 $ T $直接由虚警概率 $ P_{fa} $ 和参考单元数 $ R $确定，公式简单且易计算。这是 CA-CFAR 方法的一大特点，使其实现较为方便。

---

### SO-CFAR

在 **SO-CFAR（Smallest of CFAR）** 中，门限因子 $ T $ 需要通过特定的公式迭代计算。SO-CFAR 的虚警概率 $ P_{fa} $ 满足以下关系：

$$
P_{fa} = \sum_{i=0}^{n-1} 2 \binom{n+i-1}{i} (2 + T)^{-(n+i)}
$$

其中：
- $ n $是滑动窗口参考单元的一半个数（总参考单元数 $R = 2n $）；
- $ T $是门限因子（待求解）；

由于这种关系具有一定复杂性，$ T $通常通过数值方法（如迭代或数值求解）计算。

以下是 **SO-CFAR** 的 Python 实现，用于计算门限因子 $ T $。

In [5]:
import numpy as np

def pfa_so_cfar(T, n):
    """SO-CFAR 的 Pfa(T;n)，避免大数幂的稳定实现"""
    a = T + 2.0
    denom = a ** n                 # a^(n+i)
    c = 1.0                        # C(n-1,0)
    s = 0.0
    for i in range(n):
        s += c / denom
        denom *= a                 # 下一项的幂
        c *= (n + i) / (i + 1.0)   # 递推 C(n+i, i+1)
    return 2.0 * s

def so_cfar_T_from_pfa(pfa, n, tol=1e-10, max_iter=100):
    """给定 Pfa 与 n，数值求解门限 T"""
    # 最大可达虚警（T=0）检查
    pfa_max = pfa_so_cfar(0.0, n)
    if pfa >= pfa_max:
        raise ValueError(f"Pfa 必须小于 {pfa_max:.6g} (对应 T=0)，否则无非负解。")

    # 夹逼区间（经验近似）
    t_lo = max(0.0, (2.0 / pfa)**(1.0 / n) - 2.0)
    t_hi = (2.0 / pfa)**(1.0 / n) - 1.0

    # 若上界仍不足以使 Pfa 降到目标以下，放大上界
    while pfa_so_cfar(t_hi, n) > pfa:
        t_hi = 2.0 * t_hi + 1.0

    # 二分法
    for _ in range(max_iter):
        mid = 0.5 * (t_lo + t_hi)
        val = pfa_so_cfar(mid, n)
        if abs(val - pfa) <= tol * pfa:
            return mid
        if val > pfa:
            t_lo = mid
        else:
            t_hi = mid
    return 0.5 * (t_lo + t_hi)

# 示例：n=8，Pfa=1e-3
if __name__ == "__main__":
    n, Pfa = 8, 1e-3
    T = so_cfar_T_from_pfa(Pfa, n)
    print(f"给定Pfa={Pfa:.6g}, n={n}, 求解得到 T ≈ {T:.6f}, 验证 Pfa(T) = {pfa_so_cfar(T, n):.6g}")

给定Pfa=0.001, n=8, 求解得到 T ≈ 1.574964, 验证 Pfa(T) = 0.001


### GO-CFAR

在 **GO-CFAR（Greatest of CFAR）** 中，门限因子 $ T $ 需要通过特定的公式迭代计算。GO-CFAR 的虚警概率 $ P_{fa} $ 满足以下关系：

$$
P_{fa}^{\mathrm{GO}}(T;n)
= \frac{2}{\bigl(1+T/n\bigr)^n}
2\sum_{i=0}^{n-1}\binom{n+i-1}{i}\bigl(2+T/n\bigr)^{-(n+i)}.
$$

其中：
- $ n $是滑动窗口参考单元的一半个数（总参考单元数 $R = 2n $）；
- $ T $是门限因子（待求解）；

由于这种关系具有一定复杂性，$ T $通常通过数值方法（如迭代或数值求解）计算。

以下是 **GO-CFAR** 的 Python 实现，用于计算门限因子 $ T $。

In [5]:
import numpy as np

def _sum_comb_series(T, n):
    """S(T;n) = sum_{i=0}^{n-1} C(n+i-1,i) / (T+2)^{n+i}，稳定递推"""
    a = T + 2.0
    denom = a ** n
    c = 1.0  # C(n-1,0)
    s = 0.0
    for i in range(n):
        s += c / denom
        denom *= a
        c *= (n + i) / (i + 1.0)  # C(n+i, i+1)
    return s

def pfa_so_cfar(T, n):
    return 2.0 * _sum_comb_series(T, n)

def pfa_go_cfar(T, n):
    A = (1.0 + T) ** (-n)
    S = _sum_comb_series(T, n)
    val = 2.0 * (A - S)
    # 数值误差保护
    return max(val, 0.0)

def go_cfar_T_from_pfa(pfa, n, tol=1e-10, max_iter=100):
    """给定 Pfa 与 n，求 GO-CFAR 门限 T（二分法）"""
    if n == 1:
        # 闭式解
        if not (0 < pfa < 1.0):
            raise ValueError("Pfa 必须在 (0,1) 内。")
        return (-3.0 + np.sqrt(1.0 + 8.0 / pfa)) / 2.0

    # 可达最大虚警（T=0）检查
    pfa_max = pfa_go_cfar(0.0, n)
    if not (0 < pfa < pfa_max):
        raise ValueError(f"Pfa 必须在 (0, {pfa_max:.6g}) 内，当前={pfa:g}")

    # 夹逼区间：下界 0；上界用 2/(1+T)^n 的上界反解
    t_lo = 0.0
    t_hi = (2.0 / pfa) ** (1.0 / n) - 1.0
    # 保证上界确实使 Pfa <= 目标；否则扩大
    while pfa_go_cfar(t_hi, n) > pfa:
        t_hi = 2.0 * t_hi + 1.0

    # 二分
    for _ in range(max_iter):
        mid = 0.5 * (t_lo + t_hi)
        val = pfa_go_cfar(mid, n)
        if abs(val - pfa) <= tol * pfa:
            return mid
        if val > pfa:
            t_lo = mid
        else:
            t_hi = mid
    return 0.5 * (t_lo + t_hi)

# 示例：n=8，Pfa=1e-3
if __name__ == "__main__":
    n, Pfa = 4, 1e-3
    T = go_cfar_T_from_pfa(Pfa, n)
    print(f"给定Pfa={Pfa:.6g}, n={n}, T ≈ {T:.6f}, 验证 Pfa_GO(T) = {pfa_go_cfar(T, n):.6g}")

给定Pfa=0.001, n=4, T ≈ 2.267497, 验证 Pfa_GO(T) = 0.001
